In [2]:
import numpy as np
import utils
from env_map import env
from agent import agent
from sarsa import SARSA
from q_learning import QLearning
from datetime import datetime

FNAMEs = ['map1', 'map2', 'map3', 'map4']
EPSILONs = [0.2]
GAMMAs = [0.89]
NUM_EPISODES = 500
NUM_STEPS = 200
EPS = 0.1
LR = 0.1

ABSTRACTION_SIZ = (20,20)
TARGET_POS = (2, 12)
START_POS = (12, 2)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
#abstract maps
maps = {}
for f in FNAMEs:
    dim_red = ABSTRACTION_SIZ
    if f == 'map1':
        dim_red = (1,1)
    surface, pooled_map = utils.bmp_to_mat(dim_red, f)
    maps[f] = pooled_map

## Complexity of Map
- [ ] SARSA | map1
- [ ] SARSA | map2
- [ ] SARSA | map3
- [ ] SARSA | map4
- [ ] Q-Learning | map1
- [ ] Q-Learning | map2
- [ ] Q-Learning | map3
- [ ] Q-Learning | map4

In [5]:
#get template dict
e_group = 'com'
com = utils.get_results_dict(e_group, FNAMEs)

#train an agent on each map
for map_name in maps:
    ts = int(datetime.now().timestamp())  
    m = maps[map_name]

    #initialize RL agent and environment for SARSA
    env_ = env(m, m.shape, 's1', START_POS, TARGET_POS)
    agent_ = agent(env_, START_POS)
    sarsa = SARSA(env_, agent_, NUM_EPISODES, NUM_STEPS,
                  EPSILONs[0], LR, GAMMAs[0])
    #train agent using sarsa
    episode_rewards, traj, time_cost_s, steps_taken = sarsa.run()

    #track the agent fastest successful trajectory
    if steps_taken:
        sp_episode = min(steps_taken, key=steps_taken.get)
        sp_val = min(steps_taken.values())
    else:
        sp_episode = max(traj.keys())
        sp_val = len(traj[sp_episode])

    #save results
    com[map_name]['timestamp'].append(ts)
    com[map_name]['num_episodes'].append(NUM_EPISODES)
    com[map_name]['num_steps'].append(NUM_STEPS)
    com[map_name]['method'].append('SARSA')
    com[map_name]['time_cost'].append(time_cost_s)
    com[map_name]['shortest_path'].append((sp_episode, sp_val))
    com[map_name]['reward_sequence'].append(episode_rewards)
    com[map_name]['final_q_table'].append(agent_.q_table)
    
    utils.log(agent_.q_table, f'policies/{e_group}/{map_name}/{ts}/sarsa_{map_name}.csv')

    #initialize RL agent and environment for QLearning
    env_q = env(m, m.shape, 's1', START_POS, TARGET_POS)
    agent_q = agent(env_q, START_POS)
    q_learning = QLearning(env_q, agent_q, NUM_EPISODES, NUM_STEPS,
                           EPSILONs[0], LR, GAMMAs[0])
    #train agent using QLearning
    episode_rewards_q, traj_q, time_cost_q, steps_taken_q = q_learning.run()

    #track the agent fastest successful trajectory
    if steps_taken_q:
        sp_episode_q = min(steps_taken_q, key=steps_taken_q.get)
        sp_val_q = min(steps_taken_q.values())
    else:
        sp_episode_q = max(traj_q.keys())
        sp_val_q = len(traj_q[sp_episode_q])

    com[map_name]['timestamp'].append(ts)
    com[map_name]['num_episodes'].append(NUM_EPISODES)
    com[map_name]['num_steps'].append(NUM_STEPS)
    com[map_name]['method'].append('QLearning')
    com[map_name]['time_cost'].append(time_cost_q)
    com[map_name]['shortest_path'].append((sp_episode_q, sp_val_q))
    com[map_name]['reward_sequence'].append(episode_rewards_q)
    com[map_name]['final_q_table'].append(agent_q.q_table)
    
    utils.log(agent_q.q_table, f'policies/{e_group}/{map_name}/{ts}/qlearning_{map_name}.csv')

    utils.log(com[map_name], f'logs/{e_group}/{map_name}/{ts}/{map_name}.csv')

    #animate the best trajectory from both methods
    utils.animate(map_=m, w=m.shape[0], h=m.shape[1],
                  start=START_POS, target=TARGET_POS,
                  traj=traj[sp_episode])
    utils.animate(map_=m, w=m.shape[0], h=m.shape[1],
                  start=START_POS, target=TARGET_POS,
                  traj=traj_q[sp_episode_q])


KeyboardInterrupt



In [ ]:
'''
Complexity of Map: Testing
'''


## Exploration Rate (map4, γ=0.5)
- [ ] SARSA | ε=0
- [ ] SARSA | ε=0.5
- [ ] SARSA | ε=1
- [ ] Q-Learning | ε=0
- [ ] Q-Learning | ε=0.5
- [ ] Q-Learning | ε=1

## Discount Value (map4, ε=0.5)
- [ ] SARSA | γ=0.1
- [ ] SARSA | γ=0.5
- [ ] SARSA | γ=1
- [ ] Q-Learning | γ=0.1
- [ ] Q-Learning | γ=0.5
- [ ] Q-Learning | γ=1

## Reward Strategy (map4, best ε and γ from above)
- [ ] SARSA | S1
- [ ] SARSA | S2
- [ ] Q-Learning | S1
- [ ] Q-Learning | S2